<a href="https://colab.research.google.com/github/Otsebolu/Gen_AI_projects/blob/main/Bank_customer_service.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
%matplotlib inline
import nltk
from nltk.corpus import stopwords
from wordcloud import WordCloud
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import string
from transformers import pipeline
from datasets import Dataset
import transformers
from transformers import AutoModelForSequenceClassification
from transformers import TFAutoModelForSequenceClassification
from transformers import AutoTokenizer, AutoConfig
from transformers import TrainingArguments, Trainer
from scipy.special import softmax

In [ ]:
df = pd.read_csv('Bank_customer_service.csv', encoding='ISO-8859-1')

In [ ]:
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----  
 0   customer_review  5000 non-null   object
 1   category         5000 non-null   object
dtypes: object(2)
memory usage: 78.2+ KB


None

In [ ]:
df.rename(columns={'customer_review': 'text', 'category': 'label'}, inplace=True)


In [ ]:
print(df.isnull().sum())
df.head()

text     5000
label    5000
dtype: int64


,text,label
0,I am not happy with the services,not_happy
1,The bank is very good.,happy
2,I am happy with the services,happy
3,I am not satisfied with the services,not_happy
4,The customer service is great,happy


In [ ]:
df['label'].value_counts()

label
happy        2500
not_happy    2500
Name: count, dtype: int64


In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['label'] = le.fit_transform(df['label'])


In [ ]:
df.head()

,text,label
0,I am not happy with the services,1
1,The bank is very good.,0
2,I am happy with the services,0
3,I am not satisfied with the services,1
4,The customer service is great,0


In [ ]:
df.loc[df['label']==0, 'label'] = 'happy'
df.loc[df['label']==1, 'label'] = 'not_happy'


In [ ]:
df['label'].value_counts().plot(kind='bar')


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
sns.countplot(x='label', data=df, ax=ax1)
ax1.set_title('Label distribution')
df['label'].value_counts().plot(kind='pie', ax=ax2, autopct='%1.1f%%')
ax2.set_title('Label distribution')
plt.show()


In [ ]:
def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

In [ ]:
df['text'] = df['text'].apply(remove_punctuation)

In [ ]:
df.label.value_counts()
df.info()


happy        5000
not_happy    5000
Name: count, dtype: int64


text     5000
label    5000
dtype: int64

In [ ]:
nltk.download('stopwords')

In [ ]:
stop_words = set(stopwords.words('english'))
def remove_stopwords(text):
    return ' '.join([word for word in text.split() if word.lower() not in stop_words])


In [ ]:
df['text'] = df['text'].apply(remove_stopwords)

In [ ]:
df['text'].str.lower()


In [ ]:
text = df['text'].str.cat(sep=' ')
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(text)


In [ ]:
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud of Text Data')
plt.show()


In [ ]:
tf = TfidfVectorizer(max_features=1000)
X = tf.fit_transform(df['text'])
y = df['label']


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
model = LogisticRegression()
model.fit(X_train, y_train)


In [ ]:
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print(f"Accuracy: {accuracy}")
print("Classification Report:\n", report)
print("Confusion Matrix:\n", cm)


Accuracy: 0.999
Classification Report:
              precision    recall  f1-score   support

       happy       1.00      1.00      1.00       513
   not_happy       1.00      1.00      1.00       487

    accuracy                           1.00      1000
   macro avg       1.00      1.00      1.00      1000
weighted avg       1.00      1.00      1.00      1000

Confusion Matrix:
[[513   0]
 [  1 486]]


In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Happy', 'Not Happy'], yticklabels=['Happy', 'Not Happy'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
new_reviews = pd.Series(["The bank is not good", "The services are amazing"])
new_reviews_transformed = tf.transform(new_reviews)
predictions = model.predict(new_reviews_transformed)

for review, prediction in zip(new_reviews, predictions):
    print(f"Review: {review}, Predicted: {prediction}")


In [ ]:
def analyze_review(text):
    text_transformed = tf.transform([text])
    prediction = model.predict(text_transformed)[0]
    return prediction


In [ ]:
transformers.utils.notebook.huggingface_hub.snapshot_download(repo_id='distilbert-base-uncased', allow_patterns=['config.json', 'vocab.txt', 'tokenizer_config.json', 'tokenizer.json'])


In [ ]:
model_name = 'distilbert-base-uncased'
config = AutoConfig.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model_distilbert = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)


In [ ]:
class_names = ['happy', 'not_happy']


In [ ]:
def preprocess_function(examples):
    return tokenizer(examples['text'], truncation=True, padding=True)


In [ ]:
dataset = Dataset.from_pandas(df)
tokenized_dataset = dataset.map(preprocess_function, batched=True)


In [ ]:
train_test_split = tokenized_dataset.train_test_split(test_size=0.2, seed=42)
print(train_test_split)

DatasetDict({})


In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
)

In [ ]:
trainer = Trainer(
    model=model_distilbert,
    args=training_args,
    train_dataset=train_test_split['train'],
    eval_dataset=train_test_split['test'],
)


In [ ]:
trainer.train()


TrainOutput(global_step=1500, training_loss=0.014299955745140712, metrics={'train_runtime': 171.1852, 'train_samples_per_second': 23.366, 'train_steps_per_second': 2.921, 'total_flos': 52932455987200.0, 'train_loss': 0.014299955745140712, 'epoch': 3.0})


In [ ]:
trainer.evaluate()

{'eval_loss': 0.00994998786598444, 'eval_runtime': 2.3683, 'eval_samples_per_second': 422.259, 'eval_steps_per_second': 52.731, 'epoch': 3.0}

In [ ]:
test_sentences = ["The bank is not good", "The services are amazing"]
for sentence in test_sentences:
    inputs = tokenizer(sentence, return_tensors='pt')
    outputs = model_distilbert(**inputs)
    scores = outputs.logits[0].detach().numpy()
    scores = softmax(scores)
    prediction = class_names[np.argmax(scores)]
    print(f"Review: {sentence}, Prediction: {prediction}")

Review: The bank is not good, Prediction: not_happy
Review: The services are amazing, Prediction: happy
